# Session 13 — Your turn: Transformers beyond text

Count what a vision transformer sees, then do zero-shot classification the way CLIP does: pick the class whose text vector is most similar to the image vector.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: Patches are tokens

Write `vit_tokens(image_size, patch_size)` returning the number of patch tokens (`(image_size // patch_size)**2`) plus one class token, and `patch_embedding_params(patch_size, channels, width)` for the linear layer that turns one patch into a vector (weights plus bias).
Use them for a 224 x 224 RGB image with 16 x 16 patches and width 768: `n_tokens`, `embed_params`. Then `n_tokens_small` for 32 x 32 patches, and `attention_pairs_ratio`, how many times more token pairs the 16 x 16 version has.

In [2]:
def vit_tokens(image_size, patch_size):
    return (image_size // patch_size) ** 2 + 1


def patch_embedding_params(patch_size, channels, width):
    return patch_size * patch_size * channels * width + width


n_tokens = vit_tokens(224, 16)
embed_params = patch_embedding_params(16, 3, 768)
n_tokens_small = vit_tokens(224, 32)
attention_pairs_ratio = n_tokens ** 2 / n_tokens_small ** 2
print(n_tokens, embed_params, n_tokens_small, attention_pairs_ratio)

197 590592 50 15.5236


In [3]:
check("196 patches and a class token: 197 tokens", lambda: n_tokens == 197)
check("the patch embedding has 590,592 parameters", lambda: embed_params == 590_592)
check("32 x 32 patches give 50 tokens", lambda: n_tokens_small == 50)
check("attention over 197 tokens costs about 15 times more pairs than over 50", lambda: close(attention_pairs_ratio, 15.5, 0.1))
summary()

PASS  196 patches and a class token: 197 tokens
PASS  the patch embedding has 590,592 parameters
PASS  32 x 32 patches give 50 tokens
PASS  attention over 197 tokens costs about 15 times more pairs than over 50

4 of 4 checks passed.  Well done.


**Why it works.** An image becomes a sequence by cutting it into patches, each flattened and projected into one token. Halving the patch side quadruples the tokens and multiplies the attention pairs by about sixteen, which is why small patches are expensive.

## Task 2: Zero-shot classification by similarity

`text_vectors` holds one vector per class name (`classes`), `image_vectors` one per image, and `labels` the true class of each image. Normalise all vectors to length 1, compute the cosine similarity of each image to each class text, predict the most similar class (`predicted`), and store the `accuracy`.
Then give the same classification with a **prompt shift**: `text_vectors_shifted` is what you would get from a badly worded prompt; compute `accuracy_shifted` the same way.

In [4]:
rng = np.random.default_rng(0)
classes = ["daisy", "rose", "tulip"]
text_vectors = rng.normal(size=(3, 16))
labels = rng.integers(0, 3, 90)
image_vectors = text_vectors[labels] * 0.6 + rng.normal(size=(90, 16))
text_vectors_shifted = text_vectors + 2.0 * rng.normal(size=(3, 16))

def classify(images, texts):
    i = images / np.linalg.norm(images, axis=1, keepdims=True)
    t = texts / np.linalg.norm(texts, axis=1, keepdims=True)
    return (i @ t.T).argmax(axis=1)


predicted = classify(image_vectors, text_vectors)
accuracy = float((predicted == labels).mean())
accuracy_shifted = float((classify(image_vectors, text_vectors_shifted) == labels).mean())
print(accuracy, accuracy_shifted)

0.9 0.6666666666666666


In [5]:
check("one prediction per image", lambda: np.shape(predicted) == (90,))
check("with good text vectors the accuracy is well above chance (1/3)", lambda: accuracy > 0.6)
check("a badly worded prompt costs accuracy", lambda: accuracy_shifted < accuracy - 0.1)
check("classify uses cosine similarity: scaling the images does not change the answer", lambda: np.array_equal(classify(image_vectors * 5, text_vectors), predicted))
summary()

PASS  one prediction per image
PASS  with good text vectors the accuracy is well above chance (1/3)
PASS  a badly worded prompt costs accuracy
PASS  classify uses cosine similarity: scaling the images does not change the answer

4 of 4 checks passed.  Well done.


**Why it works.** CLIP never sees class labels while training; it learns to put an image near the text that describes it, so a classifier is just 'which class name is closest'. The result depends heavily on the words you choose for the class text, which the session measures as a spread of accuracy across prompts.